In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/telco_churn.csv")
print(df.shape)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
df.info()
print(df.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [3]:

print(df["TotalCharges"].dtype)  # deve ser object (texto)

# Linhas em que não dá para converter para número
invalidos = df[pd.to_numeric(df["TotalCharges"], errors="coerce").isna()]
print(len(invalidos))
invalidos[["customerID", "tenure", "TotalCharges", "Churn"]]

str
11


,customerID,tenure,TotalCharges,Churn
488,4472-LVYGI,0,,No
753,3115-CZMZD,0,,No
936,5709-LVOEQ,0,,No
1082,4367-NUYAO,0,,No
1340,1371-DWPAZ,0,,No
3331,7644-OMVMY,0,,No
3826,3213-VVOLG,0,,No
4380,2520-SGTTA,0,,No
5218,2923-ARZLG,0,,No
6670,4075-WKNIU,0,,No


In [4]:
contagem = df["Churn"].value_counts()
proporcao = df["Churn"].value_counts(normalize=True)
print(contagem)
print(proporcao.round(3))

Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64


In [5]:
for col in ["Contract", "PaymentMethod", "InternetService"]:
    print(df.groupby(col)["Churn"].apply(lambda s: (s == "Yes").mean()).round(3))
    print()

df["tenure_faixa"] = pd.cut(df["tenure"], [-1, 6, 12, 24, 48, 72])
print(df.groupby("tenure_faixa", observed=True)["Churn"].apply(lambda s: (s == "Yes").mean()).round(3))

Contract
Month-to-month    0.427
One year          0.113
Two year          0.028
Name: Churn, dtype: float64

PaymentMethod
Bank transfer (automatic)    0.167
Credit card (automatic)      0.152
Electronic check             0.453
Mailed check                 0.191
Name: Churn, dtype: float64

InternetService
DSL            0.190
Fiber optic    0.419
No             0.074
Name: Churn, dtype: float64

tenure_faixa
(-1, 6]     0.529
(6, 12]     0.359
(12, 24]    0.287
(24, 48]    0.204
(48, 72]    0.095
Name: Churn, dtype: float64


# Passo 4: Calcular o baseline

In [6]:
y = (df["Churn"] == "Yes").astype(int)

acuracia_baseline = 1 - y.mean()
print(f"Acurácia do baseline (sempre 'não cancela'): {acuracia_baseline:.3f}")
print(f"Recall do baseline: 0.0 (não encontra nenhum churner)")

Acurácia do baseline (sempre 'não cancela'): 0.735
Recall do baseline: 0.0 (não encontra nenhum churner)


In [7]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, average_precision_score

dummy = DummyClassifier(strategy="most_frequent").fit(df, y)
pred = dummy.predict(df)

print("Acurácia:", accuracy_score(y, pred))
print("Recall:", recall_score(y, pred))
print("PR-AUC:", average_precision_score(y, dummy.predict_proba(df)[:, 1]))

Acurácia: 0.7346301292063041
Recall: 0.0
PR-AUC: 0.2653698707936959
